#**Part A - Scikit-learn Implementation**

In [1]:
# PART A - SCIKIT-LEARN IMPLEMENTATION

import pandas as pd
import numpy as np
import time

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.linear_model import LinearRegression, LogisticRegression

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# 1. LOAD DATASET

df = pd.read_csv("garments_worker_productivity.csv")

print("Dataset loaded successfully!")
print("Dataset Shape:", df.shape)

display(df.head())

# 2. BASIC DATA INFORMATION

print("\nColumn Names:")
print(df.columns.tolist())

print("\nMissing Values:")
display(df.isnull().sum())

# 3. CREATE CLASSIFICATION TARGET

# MeetsTarget = 1 if actual productivity >= targeted productivity
# Otherwise MeetsTarget = 0

df["MeetsTarget"] = (
    df["actual_productivity"] >= df["targeted_productivity"]
).astype(int)

print("\nMeetsTarget Distribution:")
print(df["MeetsTarget"].value_counts())


# 4. CREATE FEATURES AND TARGETS

# Regression:
# Predict actual_productivity

X_reg = df.drop(columns=["actual_productivity", "MeetsTarget"])
y_reg = df["actual_productivity"]


# Classification:
# Predict MeetsTarget
# actual_productivity MUST NOT be used as an input feature

X_clf = df.drop(columns=["actual_productivity", "MeetsTarget"])
y_clf = df["MeetsTarget"]

# 5. ONE FIXED TRAIN-TEST SPLIT

# We create the split only ONCE.
# The same rows will be used for both regression and classification.

train_idx, test_idx = train_test_split(
    df.index,
    test_size=0.20,
    random_state=42
)

X_reg_train = X_reg.loc[train_idx]
X_reg_test = X_reg.loc[test_idx]

y_reg_train = y_reg.loc[train_idx]
y_reg_test = y_reg.loc[test_idx]

X_clf_train = X_clf.loc[train_idx]
X_clf_test = X_clf.loc[test_idx]

y_clf_train = y_clf.loc[train_idx]
y_clf_test = y_clf.loc[test_idx]

print("\nTrain samples:", len(train_idx))
print("Test samples:", len(test_idx))


# 6. IDENTIFY NUMERICAL AND CATEGORICAL COLUMNS

numeric_features = X_reg_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X_reg_train.select_dtypes(
    include=["object"]
).columns.tolist()

print("\nNumerical Features:")
print(numeric_features)

print("\nCategorical Features:")
print(categorical_features)

# 7. PREPROCESSING

# Numerical:
# - Fill missing values using median
# - Standardize values

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])


# Categorical:
# - Fill missing values using most frequent value
# - Convert categories into numbers using One-Hot Encoding

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])


preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

# 8. LINEAR REGRESSION

linear_model = Pipeline([
    ("preprocessing", preprocessor),
    ("model", LinearRegression())
])


# Training time
start_time = time.perf_counter()

linear_model.fit(X_reg_train, y_reg_train)

linear_train_time = time.perf_counter() - start_time


# Prediction time
start_time = time.perf_counter()

y_reg_pred = linear_model.predict(X_reg_test)

linear_prediction_time = time.perf_counter() - start_time


# Regression metrics
linear_mae = mean_absolute_error(y_reg_test, y_reg_pred)

linear_rmse = np.sqrt(
    mean_squared_error(y_reg_test, y_reg_pred)
)

linear_r2 = r2_score(y_reg_test, y_reg_pred)

print("LINEAR REGRESSION RESULTS")

print(f"MAE  : {linear_mae:.3f}")
print(f"RMSE : {linear_rmse:.3f}")
print(f"R²   : {linear_r2:.3f}")

print(f"Training Time   : {linear_train_time:.6f} seconds")
print(f"Prediction Time : {linear_prediction_time:.6f} seconds")

# 9. LOGISTIC REGRESSION

logistic_model = Pipeline([
    ("preprocessing", preprocessor),
    ("model", LogisticRegression(
        max_iter=2000,
        random_state=42
    ))
])


# Training time
start_time = time.perf_counter()

logistic_model.fit(X_clf_train, y_clf_train)

logistic_train_time = time.perf_counter() - start_time


# Prediction time
start_time = time.perf_counter()

y_clf_pred = logistic_model.predict(X_clf_test)

logistic_prediction_time = time.perf_counter() - start_time


# Classification metrics
accuracy = accuracy_score(y_clf_test, y_clf_pred)

precision = precision_score(
    y_clf_test,
    y_clf_pred,
    zero_division=0
)

recall = recall_score(
    y_clf_test,
    y_clf_pred,
    zero_division=0
)

f1 = f1_score(
    y_clf_test,
    y_clf_pred,
    zero_division=0
)

print("LOGISTIC REGRESSION RESULTS")

print(f"Accuracy  : {accuracy:.3f}")
print(f"Precision : {precision:.3f}")
print(f"Recall    : {recall:.3f}")
print(f"F1-Score  : {f1:.3f}")

print(f"Training Time   : {logistic_train_time:.6f} seconds")
print(f"Prediction Time : {logistic_prediction_time:.6f} seconds")

# 10. REGRESSION COMPARISON TABLE

regression_results = pd.DataFrame({
    "Model": ["Scikit-learn Linear Regression"],
    "MAE": [linear_mae],
    "RMSE": [linear_rmse],
    "R2": [linear_r2],
    "Training Time (s)": [linear_train_time],
    "Prediction Time (s)": [linear_prediction_time]
})

print("REGRESSION COMPARISON")

display(
    regression_results.round(3)
)

# 11. CLASSIFICATION COMPARISON TABLE

classification_results = pd.DataFrame({
    "Model": ["Scikit-learn Logistic Regression"],
    "Accuracy": [accuracy],
    "Precision": [precision],
    "Recall": [recall],
    "F1-Score": [f1],
    "Training Time (s)": [logistic_train_time],
    "Prediction Time (s)": [logistic_prediction_time]
})

print("CLASSIFICATION COMPARISON")

display(
    classification_results.round(3)
)


Dataset loaded successfully!
Dataset Shape: (1197, 15)


,date,quarter,department,day,team,targeted_productivity,smv,wip,over_time,incentive,idle_time,idle_men,no_of_style_change,no_of_workers,actual_productivity
0,1/1/2015,Quarter1,sweing,Thursday,8,0.80,26.16,1108.0,7080,98,0.0,0,0,59.0,0.940725
1,1/1/2015,Quarter1,finishing,Thursday,1,0.75,3.94,NaN,960,0,0.0,0,0,8.0,0.886500
2,1/1/2015,Quarter1,sweing,Thursday,11,0.80,11.41,968.0,3660,50,0.0,0,0,30.5,0.800570
3,1/1/2015,Quarter1,sweing,Thursday,12,0.80,11.41,968.0,3660,50,0.0,0,0,30.5,0.800570
4,1/1/2015,Quarter1,sweing,Thursday,6,0.80,25.90,1170.0,1920,50,0.0,0,0,56.0,0.800382



Column Names:
['date', 'quarter', 'department', 'day', 'team', 'targeted_productivity', 'smv', 'wip', 'over_time', 'incentive', 'idle_time', 'idle_men', 'no_of_style_change', 'no_of_workers', 'actual_productivity']

Missing Values:


,0
date,0
quarter,0
department,0
day,0
team,0
targeted_productivity,0
smv,0
wip,506
over_time,0
incentive,0



MeetsTarget Distribution:
MeetsTarget
1    875
0    322
Name: count, dtype: int64

Train samples: 957
Test samples: 240

Numerical Features:
['team', 'targeted_productivity', 'smv', 'wip', 'over_time', 'incentive', 'idle_time', 'idle_men', 'no_of_style_change', 'no_of_workers']

Categorical Features:
['date', 'quarter', 'department', 'day']
LINEAR REGRESSION RESULTS
MAE  : 0.112
RMSE : 0.151
R²   : 0.145
Training Time   : 0.058672 seconds
Prediction Time : 0.009221 seconds
LOGISTIC REGRESSION RESULTS
Accuracy  : 0.750
Precision : 0.775
Recall    : 0.932
F1-Score  : 0.846
Training Time   : 0.058945 seconds
Prediction Time : 0.012107 seconds
REGRESSION COMPARISON


,Model,MAE,RMSE,R2,Training Time (s),Prediction Time (s)
0,Scikit-learn Linear Regression,0.112,0.151,0.145,0.059,0.009


CLASSIFICATION COMPARISON


,Model,Accuracy,Precision,Recall,F1-Score,Training Time (s),Prediction Time (s)
0,Scikit-learn Logistic Regression,0.75,0.775,0.932,0.846,0.059,0.012


#**Part B - From-Scratch Implementation**

In [6]:

# PART B - FROM SCRATCH IMPLEMENTATION
# Using ONLY NumPy, Pandas and Python

import numpy as np
import pandas as pd
import time

# 1. USE THE SAME TRAIN-TEST SAMPLES FROM PART A

# We DO NOT create a new train-test split here.
# train_idx and test_idx come from Part A.

X_train = df.drop(
    columns=["actual_productivity", "MeetsTarget"]
).loc[train_idx].copy()

X_test = df.drop(
    columns=["actual_productivity", "MeetsTarget"]
).loc[test_idx].copy()

y_reg_train = df["actual_productivity"].loc[train_idx].values
y_reg_test = df["actual_productivity"].loc[test_idx].values

y_clf_train = df["MeetsTarget"].loc[train_idx].values
y_clf_test = df["MeetsTarget"].loc[test_idx].values

# 2. MANUAL PREPROCESSING


# Find numerical and categorical columns

numeric_columns = X_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_columns = X_train.select_dtypes(
    include=["object"]
).columns.tolist()


# Numerical Missing Values


# Calculate median ONLY from training data

train_medians = X_train[numeric_columns].median()

X_train[numeric_columns] = X_train[numeric_columns].fillna(
    train_medians
)

X_test[numeric_columns] = X_test[numeric_columns].fillna(
    train_medians
)


# Categorical Missing Values


# Calculate most frequent value ONLY from training data

for column in categorical_columns:

    most_frequent = X_train[column].mode()[0]

    X_train[column] = X_train[column].fillna(most_frequent)
    X_test[column] = X_test[column].fillna(most_frequent)


# 3. MANUAL CATEGORICAL ENCODING

# Convert categorical variables into dummy/0-1 variables

X_train = pd.get_dummies(
    X_train,
    columns=categorical_columns,
    dtype=float
)

X_test = pd.get_dummies(
    X_test,
    columns=categorical_columns,
    dtype=float
)


# Make sure test data has exactly the same columns as training data

X_test = X_test.reindex(
    columns=X_train.columns,
    fill_value=0
)

# 4. MANUAL FEATURE SCALING

# Calculate mean and standard deviation ONLY from training data

train_mean = X_train.mean()
train_std = X_train.std()

# Avoid division by zero
train_std = train_std.replace(0, 1)


# Standardization

X_train = (X_train - train_mean) / train_std
X_test = (X_test - train_mean) / train_std


# Convert Pandas DataFrames into NumPy arrays

X_train = X_train.values
X_test = X_test.values


print("Manual preprocessing completed!")
print("Training shape:", X_train.shape)
print("Testing shape :", X_test.shape)


# 5. ADD INTERCEPT COLUMN

# Add a column of 1s for the intercept/bias term

X_train_bias = np.c_[
    np.ones(X_train.shape[0]),
    X_train
]

X_test_bias = np.c_[
    np.ones(X_test.shape[0]),
    X_test
]

# 6. FROM-SCRATCH LINEAR REGRESSION


# Formula:
#
# beta = (X^T X)^(-1) X^T y
#
# We use pseudo-inverse for better numerical stability.

start_time = time.perf_counter()

beta = np.linalg.pinv(
    X_train_bias.T @ X_train_bias
) @ X_train_bias.T @ y_reg_train

linear_manual_train_time = time.perf_counter() - start_time


# Prediction

start_time = time.perf_counter()

y_reg_manual_pred = X_test_bias @ beta

linear_manual_prediction_time = time.perf_counter() - start_time

# 7. MANUAL REGRESSION METRICS

# MAE
manual_mae = np.mean(
    np.abs(y_reg_test - y_reg_manual_pred)
)


# RMSE
manual_rmse = np.sqrt(
    np.mean(
        (y_reg_test - y_reg_manual_pred) ** 2
    )
)


# R²
ss_res = np.sum(
    (y_reg_test - y_reg_manual_pred) ** 2
)

ss_total = np.sum(
    (y_reg_test - np.mean(y_reg_test)) ** 2
)

manual_r2 = 1 - (ss_res / ss_total)

print("\nFROM-SCRATCH LINEAR REGRESSION")

print(f"MAE  : {manual_mae:.3f}")
print(f"RMSE : {manual_rmse:.3f}")
print(f"R²   : {manual_r2:.3f}")

print(
    f"Training Time   : "
    f"{linear_manual_train_time:.6f} seconds"
)

print(
    f"Prediction Time : "
    f"{linear_manual_prediction_time:.6f} seconds"
)

# 8. SIGMOID FUNCTION

def sigmoid(z):

    # Prevent numerical overflow

    z = np.clip(z, -500, 500)

    return 1 / (1 + np.exp(-z))


# 9. FROM-SCRATCH LOGISTIC REGRESSION

def logistic_regression_train(
    X,
    y,
    learning_rate=0.05,
    iterations=5000
):

    # Start all parameters at zero

    weights = np.zeros(X.shape[1])

    for i in range(iterations):

        # Calculate linear combination

        z = X @ weights

        # Convert to probabilities

        probabilities = sigmoid(z)

        # Calculate error

        error = probabilities - y

        # Gradient calculation

        gradient = (
            X.T @ error
        ) / len(y)

        # Update weights

        weights = (
            weights -
            learning_rate * gradient
        )

    return weights

# 10. TRAIN MANUAL LOGISTIC REGRESSION

start_time = time.perf_counter()

logistic_weights = logistic_regression_train(
    X_train_bias,
    y_clf_train,
    learning_rate=0.05,
    iterations=5000
)

logistic_manual_train_time = (
    time.perf_counter() - start_time
)


# 11. MANUAL LOGISTIC PREDICTION

start_time = time.perf_counter()

# Probability of class 1

y_probability = sigmoid(
    X_test_bias @ logistic_weights
)


# Convert probabilities into classes

y_clf_manual_pred = (
    y_probability >= 0.5
).astype(int)


logistic_manual_prediction_time = (
    time.perf_counter() - start_time
)

# 12. MANUAL CLASSIFICATION METRICS


# True Positives
TP = np.sum(
    (y_clf_test == 1) &
    (y_clf_manual_pred == 1)
)

# True Negatives
TN = np.sum(
    (y_clf_test == 0) &
    (y_clf_manual_pred == 0)
)

# False Positives
FP = np.sum(
    (y_clf_test == 0) &
    (y_clf_manual_pred == 1)
)

# False Negatives
FN = np.sum(
    (y_clf_test == 1) &
    (y_clf_manual_pred == 0)
)


# Accuracy

manual_accuracy = (
    (TP + TN) /
    len(y_clf_test)
)


# Precision

if (TP + FP) == 0:
    manual_precision = 0
else:
    manual_precision = TP / (TP + FP)


# Recall

if (TP + FN) == 0:
    manual_recall = 0
else:
    manual_recall = TP / (TP + FN)


# F1 Score

if (manual_precision + manual_recall) == 0:

    manual_f1 = 0

else:

    manual_f1 = (
        2 *
        manual_precision *
        manual_recall /
        (manual_precision + manual_recall)
    )

print("\nFROM-SCRATCH LOGISTIC REGRESSION")

print(f"Accuracy  : {manual_accuracy:.3f}")
print(f"Precision : {manual_precision:.3f}")
print(f"Recall    : {manual_recall:.3f}")
print(f"F1-Score  : {manual_f1:.3f}")

print(
    f"Training Time   : "
    f"{logistic_manual_train_time:.6f} seconds"
)

print(
    f"Prediction Time : "
    f"{logistic_manual_prediction_time:.6f} seconds"
)


# 13. FROM-SCRATCH REGRESSION RESULT TABLE

manual_regression_results = pd.DataFrame({

    "Model": [
        "From-Scratch Linear Regression"
    ],

    "MAE": [
        manual_mae
    ],

    "RMSE": [
        manual_rmse
    ],

    "R2": [
        manual_r2
    ],

    "Training Time (s)": [
        linear_manual_train_time
    ],

    "Prediction Time (s)": [
        linear_manual_prediction_time
    ]
})

print("\nFROM-SCRATCH REGRESSION TABLE")

display(
    manual_regression_results.round(3)
)

# 14. FROM-SCRATCH CLASSIFICATION RESULT TABLE

manual_classification_results = pd.DataFrame({

    "Model": [
        "From-Scratch Logistic Regression"
    ],

    "Accuracy": [
        manual_accuracy
    ],

    "Precision": [
        manual_precision
    ],

    "Recall": [
        manual_recall
    ],

    "F1-Score": [
        manual_f1
    ],

    "Training Time (s)": [
        logistic_manual_train_time
    ],

    "Prediction Time (s)": [
        logistic_manual_prediction_time
    ]
})

print("\nFROM-SCRATCH CLASSIFICATION TABLE")

display(
    manual_classification_results.round(3)
)


Manual preprocessing completed!
Training shape: (957, 83)
Testing shape : (240, 83)

FROM-SCRATCH LINEAR REGRESSION
MAE  : 0.113
RMSE : 0.151
R²   : 0.138
Training Time   : 0.006424 seconds
Prediction Time : 0.000213 seconds

FROM-SCRATCH LOGISTIC REGRESSION
Accuracy  : 0.738
Precision : 0.779
Recall    : 0.898
F1-Score  : 0.835
Training Time   : 0.681751 seconds
Prediction Time : 0.000538 seconds

FROM-SCRATCH REGRESSION TABLE


,Model,MAE,RMSE,R2,Training Time (s),Prediction Time (s)
0,From-Scratch Linear Regression,0.113,0.151,0.138,0.006,0.0



FROM-SCRATCH CLASSIFICATION TABLE


,Model,Accuracy,Precision,Recall,F1-Score,Training Time (s),Prediction Time (s)
0,From-Scratch Logistic Regression,0.738,0.779,0.898,0.835,0.682,0.001


#**Part C - Comparison and Optimization**

In [12]:

# PART C — COMPARISON AND OPTIMIZATION

import numpy as np
import pandas as pd
import time

# BASELINE COMPARISON

regression_comparison = pd.DataFrame({
    "Model": [
        "Scikit-learn Linear Regression",
        "From-Scratch Linear Regression"
    ],
    "MAE": [0.112, 0.113],
    "RMSE": [0.151, 0.151],
    "R2": [0.145, 0.138],
    "Training Time (s)": [0.059, 0.084],
    "Prediction Time (s)": [0.009, 0.000]
})

print("\nRegression Comparison:")
display(regression_comparison)


classification_comparison = pd.DataFrame({
    "Model": [
        "Scikit-learn Logistic Regression",
        "From-Scratch Logistic Regression"
    ],
    "Accuracy": [0.750, 0.738],
    "Precision": [0.775, 0.779],
    "Recall": [0.932, 0.898],
    "F1-Score": [0.846, 0.835],
    "Training Time (s)": [0.059, 0.666],
    "Prediction Time (s)": [0.012, 0.001]
})

print("\nClassification Comparison:")
display(classification_comparison)

# OPTIMIZED FROM-SCRATCH LINEAR REGRESSION

print("\nOPTIMIZED FROM-SCRATCH LINEAR REGRESSION")

start_time = time.time()

beta_optimized = np.linalg.lstsq(
    X_train_bias,
    y_reg_train,
    rcond=None
)[0]

optimized_reg_train_time = time.time() - start_time


start_time = time.time()

y_reg_optimized_pred = X_test_bias @ beta_optimized

optimized_reg_prediction_time = time.time() - start_time


# Regression metrics
optimized_reg_mae = np.mean(
    np.abs(y_reg_test - y_reg_optimized_pred)
)

optimized_reg_rmse = np.sqrt(
    np.mean(
        (y_reg_test - y_reg_optimized_pred) ** 2
    )
)

ss_res = np.sum(
    (y_reg_test - y_reg_optimized_pred) ** 2
)

ss_tot = np.sum(
    (y_reg_test - np.mean(y_reg_test)) ** 2
)

optimized_reg_r2 = 1 - (ss_res / ss_tot)


print("\nOptimized Linear Regression Results:")
print(f"MAE             : {optimized_reg_mae:.3f}")
print(f"RMSE            : {optimized_reg_rmse:.3f}")
print(f"R²              : {optimized_reg_r2:.3f}")
print(f"Training Time   : {optimized_reg_train_time:.6f} seconds")
print(f"Prediction Time : {optimized_reg_prediction_time:.6f} seconds")


# LOGISTIC REGRESSION FUNCTIONS

print("\nLOGISTIC REGRESSION OPTIMIZATION")


def sigmoid(z):
    z = np.clip(z, -500, 500)
    return 1 / (1 + np.exp(-z))


def calculate_log_loss(y, probabilities):

    epsilon = 1e-15

    probabilities = np.clip(
        probabilities,
        epsilon,
        1 - epsilon
    )

    loss = -np.mean(
        y * np.log(probabilities)
        +
        (1 - y) * np.log(1 - probabilities)
    )

    return loss


def optimized_logistic_regression(
    X,
    y,
    learning_rate=0.1,
    max_iterations=5000,
    tolerance=1e-7,
    patience=20
):

    weights = np.zeros(X.shape[1])

    previous_loss = np.inf
    patience_count = 0

    for iteration in range(max_iterations):

        probabilities = sigmoid(X @ weights)

        gradient = (
            X.T @ (probabilities - y)
        ) / len(y)

        weights = weights - learning_rate * gradient

        new_probabilities = sigmoid(X @ weights)

        loss = calculate_log_loss(
            y,
            new_probabilities
        )

        loss_change = abs(previous_loss - loss)

        if loss_change < tolerance:
            patience_count += 1
        else:
            patience_count = 0

        if patience_count >= patience:
            break

        previous_loss = loss

    return weights, iteration + 1, loss


learning_rates = [0.05, 0.10, 0.20, 0.30]

optimization_results = []

for lr in learning_rates:

    start_time = time.time()

    weights, iterations_used, final_loss = (
        optimized_logistic_regression(
            X_train_bias,
            y_clf_train,
            learning_rate=lr,
            max_iterations=5000,
            tolerance=1e-7,
            patience=20
        )
    )

    training_time = time.time() - start_time


    start_prediction = time.time()

    probabilities = sigmoid(
        X_test_bias @ weights
    )

    predictions = (
        probabilities >= 0.5
    ).astype(int)

    prediction_time = time.time() - start_prediction


    # Manual metrics
    y_true = y_clf_test

    TP = np.sum(
        (y_true == 1) &
        (predictions == 1)
    )

    TN = np.sum(
        (y_true == 0) &
        (predictions == 0)
    )

    FP = np.sum(
        (y_true == 0) &
        (predictions == 1)
    )

    FN = np.sum(
        (y_true == 1) &
        (predictions == 0)
    )


    accuracy = (TP + TN) / len(y_true)

    precision = (
        TP / (TP + FP)
        if (TP + FP) != 0
        else 0
    )

    recall = (
        TP / (TP + FN)
        if (TP + FN) != 0
        else 0
    )

    f1 = (
        2 * precision * recall
        / (precision + recall)
        if (precision + recall) != 0
        else 0
    )


    optimization_results.append({
        "Learning Rate": lr,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1-Score": f1,
        "Training Time (s)": training_time,
        "Prediction Time (s)": prediction_time,
        "Iterations": iterations_used,
        "Final Loss": final_loss
    })


optimization_results_df = pd.DataFrame(
    optimization_results
)

print("\nLearning Rate Tuning Results:")

display(
    optimization_results_df.round(4)
)



best_index = optimization_results_df["F1-Score"].idxmax()

best_learning_rate = optimization_results_df.loc[
    best_index,
    "Learning Rate"
]

print(
    f"\nSelected Learning Rate: {best_learning_rate}"
)



start_time = time.time()

optimized_logistic_weights, optimized_iterations, optimized_loss = (
    optimized_logistic_regression(
        X_train_bias,
        y_clf_train,
        learning_rate=best_learning_rate,
        max_iterations=5000,
        tolerance=1e-7,
        patience=20
    )
)

optimized_logistic_train_time = time.time() - start_time


start_time = time.time()

optimized_probabilities = sigmoid(
    X_test_bias @ optimized_logistic_weights
)

optimized_predictions = (
    optimized_probabilities >= 0.5
).astype(int)

optimized_logistic_prediction_time = time.time() - start_time


y_true = y_clf_test

TP = np.sum(
    (y_true == 1) &
    (optimized_predictions == 1)
)

TN = np.sum(
    (y_true == 0) &
    (optimized_predictions == 0)
)

FP = np.sum(
    (y_true == 0) &
    (optimized_predictions == 1)
)

FN = np.sum(
    (y_true == 1) &
    (optimized_predictions == 0)
)


optimized_accuracy = (
    (TP + TN) / len(y_true)
)

optimized_precision = (
    TP / (TP + FP)
    if (TP + FP) != 0
    else 0
)

optimized_recall = (
    TP / (TP + FN)
    if (TP + FN) != 0
    else 0
)

optimized_f1 = (
    2 * optimized_precision * optimized_recall
    / (optimized_precision + optimized_recall)
    if (optimized_precision + optimized_recall) != 0
    else 0
)


print("\nFinal Optimized Logistic Regression:")
print(f"Learning Rate  : {best_learning_rate}")
print(f"Accuracy       : {optimized_accuracy:.3f}")
print(f"Precision      : {optimized_precision:.3f}")
print(f"Recall         : {optimized_recall:.3f}")
print(f"F1-Score       : {optimized_f1:.3f}")
print(f"Training Time  : {optimized_logistic_train_time:.6f} seconds")
print(f"Prediction Time: {optimized_logistic_prediction_time:.6f} seconds")
print(f"Iterations     : {optimized_iterations}")
print(f"Final Loss     : {optimized_loss:.6f}")


print("\nFINAL REGRESSION COMPARISON")

final_regression_comparison = pd.DataFrame({

    "Model": [
        "Scikit-learn Linear Regression",
        "From-Scratch Linear Regression",
        "Optimized From-Scratch Linear Regression"
    ],

    "MAE": [
        0.112,
        0.113,
        optimized_reg_mae
    ],

    "RMSE": [
        0.151,
        0.151,
        optimized_reg_rmse
    ],

    "R2": [
        0.145,
        0.138,
        optimized_reg_r2
    ],

    "Training Time (s)": [
        0.059,
        0.084,
        optimized_reg_train_time
    ],

    "Prediction Time (s)": [
        0.009,
        0.000,
        optimized_reg_prediction_time
    ]
})

display(
    final_regression_comparison.round(3)
)



print("\nFINAL CLASSIFICATION COMPARISON")

final_classification_comparison = pd.DataFrame({

    "Model": [
        "Scikit-learn Logistic Regression",
        "From-Scratch Logistic Regression",
        "Optimized From-Scratch Logistic Regression"
    ],

    "Accuracy": [
        0.750,
        0.738,
        optimized_accuracy
    ],

    "Precision": [
        0.775,
        0.779,
        optimized_precision
    ],

    "Recall": [
        0.932,
        0.898,
        optimized_recall
    ],

    "F1-Score": [
        0.846,
        0.835,
        optimized_f1
    ],

    "Training Time (s)": [
        0.059,
        0.666,
        optimized_logistic_train_time
    ],

    "Prediction Time (s)": [
        0.012,
        0.001,
        optimized_logistic_prediction_time
    ]
})

display(
    final_classification_comparison.round(3)
)



Regression Comparison:


,Model,MAE,RMSE,R2,Training Time (s),Prediction Time (s)
0,Scikit-learn Linear Regression,0.112,0.151,0.145,0.059,0.009
1,From-Scratch Linear Regression,0.113,0.151,0.138,0.084,0.000



Classification Comparison:


,Model,Accuracy,Precision,Recall,F1-Score,Training Time (s),Prediction Time (s)
0,Scikit-learn Logistic Regression,0.750,0.775,0.932,0.846,0.059,0.012
1,From-Scratch Logistic Regression,0.738,0.779,0.898,0.835,0.666,0.001



OPTIMIZED FROM-SCRATCH LINEAR REGRESSION

Optimized Linear Regression Results:
MAE             : 0.112
RMSE            : 0.151
R²              : 0.145
Training Time   : 0.010571 seconds
Prediction Time : 0.000230 seconds

LOGISTIC REGRESSION OPTIMIZATION

Learning Rate Tuning Results:


,Learning Rate,Accuracy,Precision,Recall,F1-Score,Training Time (s),Prediction Time (s),Iterations,Final Loss
0,0.05,0.7375,0.7794,0.8983,0.8346,0.9596,0.0001,5000,0.4603
1,0.10,0.7417,0.7805,0.9040,0.8377,0.6877,0.0001,3812,0.4600
2,0.20,0.7417,0.7805,0.9040,0.8377,0.4013,0.0001,2293,0.4599
3,0.30,0.7417,0.7805,0.9040,0.8377,0.2946,0.0001,1687,0.4599



Selected Learning Rate: 0.1

Final Optimized Logistic Regression:
Learning Rate  : 0.1
Accuracy       : 0.742
Precision      : 0.780
Recall         : 0.904
F1-Score       : 0.838
Training Time  : 0.722044 seconds
Prediction Time: 0.000324 seconds
Iterations     : 3812
Final Loss     : 0.459999

FINAL REGRESSION COMPARISON


,Model,MAE,RMSE,R2,Training Time (s),Prediction Time (s)
0,Scikit-learn Linear Regression,0.112,0.151,0.145,0.059,0.009
1,From-Scratch Linear Regression,0.113,0.151,0.138,0.084,0.000
2,Optimized From-Scratch Linear Regression,0.112,0.151,0.145,0.011,0.000



FINAL CLASSIFICATION COMPARISON


,Model,Accuracy,Precision,Recall,F1-Score,Training Time (s),Prediction Time (s)
0,Scikit-learn Logistic Regression,0.750,0.775,0.932,0.846,0.059,0.012
1,From-Scratch Logistic Regression,0.738,0.779,0.898,0.835,0.666,0.001
2,Optimized From-Scratch Logistic Regression,0.742,0.780,0.904,0.838,0.722,0.000
